# W3D1 live demo: reading the card while it works

**Eight minutes, five cells, one T4.** This is the demo slot in `decks/w3d1.md`
("nvidia-smi under load"). It runs on a free Colab T4, which is the same
runtime the students open after lunch, so onsite and online show the room the
same card and any instructor can drive it with no keys and no pod.

## Before the session

Runtime -> Change runtime type -> **T4 GPU**. Then run **Cell 0** and leave it.
It installs the pins and pulls 3 GB of weights, which takes about four minutes
and must not happen in front of the room. Everything after Cell 0 is seconds.

## What each cell is for

| Cell | Shows | The sentence to say |
|---|---|---|
| 1 | an empty card | "This is the whole machine. 15 GB, nothing on it." |
| 2 | weights land | "1.5 billion parameters at two bytes each. The arithmetic is on the wall." |
| 3 | one request | "Utilisation reads 59. Is the rest of the card spare?" |
| 4 | eight requests | "Six times the work for nine points of utilisation." |
| 5 | memory stays | "The driver shows a booking, not what is in use." |

Cells 3 and 4 are the day's thesis and cell 5 sets up the afternoon's
instrument. If you are short of time, cut cell 5 and say its one sentence.

## Reference numbers

This notebook was run twice on fresh free-tier T4s on 2026-08-29:

| | batch 1 | batch 8 |
|---|---|---|
| tokens/s | 28.0, 29.4 | 211.7, 219.2 |
| utilisation | 51%, 50% | 77%, 74% |

So about seven and a half times the work while utilisation goes from about
half the card to about three quarters. Yours will differ by a few points. If
it differs by a lot, say so out loud and ask the room why; that is a better
lesson than a number that matches.

These are not the afternoon lab's numbers and are not meant to be. The lab
sweeps dtypes and context lengths with its own prompt and its own sampling
interval, and it reads 28.2 to 171.7 tok/s at util 59 to 68
(`instructor/validation/matrix-w3d1.log`). Two honest measurements of the
same card under different conditions is the correct thing for students to
see, provided you say which is which.

In [1]:
# CELL 0 - WARM THE RUNTIME. Run at the start of the session, not at the demo.
#
# Pins mirror ../../PINS.md. This is INSTALL CELL A from the week-3 scaffold:
# profiling only, no vLLM. Installing vLLM here would replace Colab's torch and
# drag numpy back to 1.26, and the model load would then die with
# "numpy.dtype size changed" - an error that looks nothing like its cause.
import subprocess
import sys

TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     f"transformers=={TRANSFORMERS_PIN}", f"accelerate=={ACCELERATE_PIN}"],
    check=True,
)

import csv
import gc
import threading
import time

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "CPU runtime: Runtime -> Change runtime type -> T4 GPU"

SAMPLES = "/content/demo_samples.csv"
_sampler = {"thread": None, "stop": None}


def _smi(fields="utilization.gpu,memory.used"):
    """One nvidia-smi reading as a list of ints."""
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    return [int(p) for p in out.split(",")]


def _sample_loop(stop, interval_s):
    with open(SAMPLES, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["util_gpu", "mem_used_mib"])
        while not stop.is_set():
            writer.writerow(_smi())
            fh.flush()
            stop.wait(interval_s)


def start_sampler(interval_s=1.0):
    """Sample the card in the background.

    1 s, not faster. nvidia-smi's utilization.gpu is itself a rolling figure
    over the driver's own sample period of roughly a second, so polling at
    250 ms returns the same reading several times and buys noise, not
    resolution. The measured generations below are long enough that 1 s still
    gives eight or more independent points.
    """
    if _sampler["thread"] and _sampler["thread"].is_alive():
        return print("sampler already running")
    stop = threading.Event()
    thread = threading.Thread(target=_sample_loop, args=(stop, interval_s), daemon=True)
    thread.start()
    _sampler.update(thread=thread, stop=stop)


def stop_sampler():
    """Stop sampling and return mean utilisation over the samples taken."""
    _sampler["stop"].set()
    _sampler["thread"].join(timeout=5)
    _sampler.update(thread=None, stop=None)
    with open(SAMPLES) as fh:
        vals = [float(r["util_gpu"]) for r in csv.DictReader(fh)]
    return sum(vals) / len(vals) if vals else 0.0


tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"   # decoder-only: pad on the left or the batch decodes garbage
_FILLER = "The data center runs many small inference requests all day. " * 40
PROMPT = tok.decode(tok(_FILLER)["input_ids"][:512])   # a fixed 512-token prompt


def run(model, batch, new_tokens=256):
    """One measured generation. Returns (tokens/s, mean utilisation)."""
    enc = tok([PROMPT] * batch, return_tensors="pt", padding=True).to("cuda")
    model.generate(**enc, max_new_tokens=8, do_sample=False)   # warm-up, not measured
    torch.cuda.synchronize()
    start_sampler()
    t0 = time.time()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    util = stop_sampler()
    generated = (out.shape[1] - enc["input_ids"].shape[1]) * batch
    return generated / elapsed, util


snapshot_download(MODEL)   # 3 GB into the runtime's cache; nothing is loaded yet
print("warm. the demo starts at cell 1.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

warm. the demo starts at cell 1.


In [2]:
# CELL 1 - the empty card.
util, mem = _smi()
print(f"utilisation {util}%   memory in use {mem} MiB of {_smi('memory.total')[0]} MiB")

utilisation 0%   memory in use 3 MiB of 15360 MiB


In [3]:
# CELL 2 - the weights land. 1.5e9 parameters x 2 bytes = about 3.1 GB.
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

# Qwen ships sampling defaults. We generate greedily, and leaving them set
# prints two warnings per call, which is not what anyone needs on a projector.
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

params = sum(p.numel() for p in model.parameters())
predicted = params * 2 / 1024**3
on_card = _smi()[1] / 1024
print(f"parameters {params/1e9:.2f}B x 2 bytes = {predicted:.2f} GB predicted")
print(f"nvidia-smi says              {on_card:.2f} GB in use")
print(f"the {on_card - predicted:.2f} GB gap is CUDA's own context on the card, "
      "not the model. cell 5 shows it again.")

parameters 1.54B x 2 bytes = 2.88 GB predicted
nvidia-smi says              3.16 GB in use
the 0.29 GB gap is CUDA's own context on the card, not the model. cell 5 shows it again.


In [4]:
# CELL 3 - one request. Watch the utilisation reading, not the throughput.
tps1, util1 = run(model, batch=1)
print(f"batch 1:  {tps1:6.1f} tokens/s   utilisation {util1:.0f}%")

batch 1:    18.1 tokens/s   utilisation 34%


In [5]:
# CELL 4 - eight requests at once. The same card, the same weights.
tps8, util8 = run(model, batch=8)
print(f"batch 8:  {tps8:6.1f} tokens/s   utilisation {util8:.0f}%")
print(f"\n{tps8/tps1:.1f}x the work for {util8-util1:.0f} points of utilisation.")
print("Utilisation says a kernel was running. It never said how much of the card it used.")

batch 8:   217.4 tokens/s   utilisation 74%

12.0x the work for 41 points of utilisation.
Utilisation says a kernel was running. It never said how much of the card it used.


In [6]:
# CELL 5 - free the model, and watch the card not give the memory back.
peak = torch.cuda.max_memory_allocated() / 1024**3
del model
gc.collect()

print(f"torch peak allocated : {peak:.2f} GB")
print(f"torch now allocated  : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"nvidia-smi still says: {_smi()[1]/1024:.2f} GB")
print("\nThe model is gone and the driver's number has not moved. nvidia-smi is")
print("reporting the allocator's pool, which is a booking, not what is in use.")

torch.cuda.empty_cache()
print(f"\nafter empty_cache()  : {_smi()[1]/1024:.2f} GB   <- the only thing that hands it back")
print("\nThis is why the lab measures with torch.cuda.max_memory_allocated() and")
print("resets between rows, instead of trusting either of these two numbers.")

torch peak allocated : 3.20 GB
torch now allocated  : 0.01 GB
nvidia-smi still says: 3.48 GB

The model is gone and the driver's number has not moved. nvidia-smi is
reporting the allocator's pool, which is a booking, not what is in use.

after empty_cache()  : 0.15 GB   <- the only thing that hands it back

This is why the lab measures with torch.cuda.max_memory_allocated() and
resets between rows, instead of trusting either of these two numbers.


In [7]:
!pip install -q bitsandbytes==0.44.*

In [8]:
!pip install -q --upgrade bitsandbytes==0.49.2

In [9]:
import csv
import subprocess
import threading
import time

GPU_SAMPLES = "/content/gpu_samples.csv"

_sampler = {
    "thread": None,
    "stop": None,
}


def _sample_loop(
    stop_event,
    path,
    interval_s,
):
    with open(
        path,
        "w",
        newline="",
    ) as fh:

        writer = csv.writer(fh)

        writer.writerow([
            "t",
            "util_gpu",
            "mem_used_mib",
        ])

        t0 = time.time()

        while not stop_event.is_set():

            out = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
            ).stdout.strip()

            parts = [
                part.strip()
                for part in out.split(",")
            ]

            if len(parts) == 2:
                writer.writerow([
                    round(time.time() - t0, 2),
                    parts[0],
                    parts[1],
                ])

                fh.flush()

            stop_event.wait(interval_s)


def start_sampler(
    path=GPU_SAMPLES,
    interval_s=2,
):
    if (
        _sampler["thread"]
        and _sampler["thread"].is_alive()
    ):
        print(
            "sampler already running; "
            "not starting a second one"
        )
        return

    stop = threading.Event()

    thread = threading.Thread(
        target=_sample_loop,
        args=(
            stop,
            path,
            interval_s,
        ),
        daemon=True,
    )

    thread.start()

    _sampler["thread"] = thread
    _sampler["stop"] = stop

    print(
        f"sampler started -> {path} "
        f"(every {interval_s}s)"
    )


def stop_sampler():

    if _sampler["stop"]:
        _sampler["stop"].set()

    if _sampler["thread"]:
        _sampler["thread"].join(
            timeout=5
        )

    _sampler["thread"] = None
    _sampler["stop"] = None

    print("sampler stopped")


def read_util_mean(
    path=GPU_SAMPLES,
):

    values = []

    with open(path) as fh:

        for row in csv.DictReader(fh):

            try:
                values.append(
                    float(row["util_gpu"])
                )

            except (
                KeyError,
                ValueError,
            ):
                pass

    if not values:
        return 0.0

    return sum(values) / len(values)


print("official GPU sampler ready")

official GPU sampler ready


In [10]:
import gc
import time
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)

tok.pad_token = tok.eos_token
tok.padding_side = "left"


def load(dtype: str):

    if dtype == "fp16":

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
        )

    if dtype == "int8":

        quantization = BitsAndBytesConfig(
            load_in_8bit=True
        )

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=quantization,
            device_map="cuda",
        )

    raise ValueError(dtype)


def make_prompt(
    context_tokens: int,
) -> str:

    base = (
        "Summarise the following text "
        "in one sentence.\n"
    )

    filler = (
        "The data center runs many small "
        "inference requests all day. " * 400
    )

    ids = tok(
        base + filler
    )["input_ids"][:context_tokens]

    return tok.decode(ids)


def resident_vram_gb() -> float:

    torch.cuda.synchronize()

    return (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )


def profile(
    model,
    dtype: str,
    context: int,
    new_tokens: int = 128,
    batch: int = 1,
):

    prompt = make_prompt(context)

    prompts = [prompt] * batch

    enc = tok(
        prompts,
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    # Warm-up
    _ = model.generate(
        **enc,
        max_new_tokens=8,
        do_sample=False,
    )

    vram = resident_vram_gb()

    start_sampler()

    t0 = time.time()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
    )

    torch.cuda.synchronize()

    elapsed = time.time() - t0

    stop_sampler()

    generated_tokens = (
        out.shape[1]
        - enc["input_ids"].shape[1]
    ) * batch

    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram, 3),
        "util_mean": round(
            read_util_mean(),
            1,
        ),
        "tokens_per_s": round(
            generated_tokens / elapsed,
            1,
        ),
    }


def free_vram():

    gc.collect()
    torch.cuda.empty_cache()


print("measurement helpers ready")

measurement helpers ready


In [11]:
rows = []

for dtype in [
    "fp16",
    "int8",
]:

    print(
        f"\nLoading {dtype} model..."
    )

    model = load(dtype)

    for context in [
        512,
        2048,
        4096,
    ]:

        print(
            f"Profiling {dtype}, "
            f"context {context}..."
        )

        row = profile(
            model,
            dtype,
            context,
        )

        print(row)

        rows.append(row)

    del model
    free_vram()


print(
    "\nFinished all measurements."
)

print(
    "Total rows:",
    len(rows),
)


Loading fp16 model...
Profiling fp16, context 512...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler stopped
{'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 37.8, 'tokens_per_s': 20.6}
Profiling fp16, context 2048...


...

sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 512, 'vram_gb': 1.805, 'util_mean': 23.1, 'tokens_per_s': 5.7}
Profiling int8, context 2048...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 27.7, 'tokens_per_s': 5.5}
Profiling int8, context 4096...


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 28.5, 'tokens_per_s': 4.9}

Finished all measurements.
Total rows: 6


In [12]:
model = load("fp16")

b1 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=1,
)

b8 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=8,
)

del model
free_vram()


print(
    "batch 1:",
    b1,
)

print(
    "batch 8:",
    b8,
)

print(
    "tokens/s ratio:",
    round(
        b8["tokens_per_s"]
        / b1["tokens_per_s"],
        2,
    ),
)

print(
    "util delta:",
    round(
        b8["util_mean"]
        - b1["util_mean"],
        1,
    ),
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler stopped


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler started -> /content/gpu_samples.csv (every 2s)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


sampler stopped
batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 41.7, 'tokens_per_s': 26.4}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 84.0, 'tokens_per_s': 208.6}
tokens/s ratio: 7.9
util delta: 42.3


In [13]:
import json


with open(
    "profile.json",
    "w",
) as file:

    json.dump(
        rows,
        file,
        indent=2,
    )


with open(
    "batch_check.json",
    "w",
) as file:

    json.dump(
        {
            "batch1_tokens_per_s":
                b1["tokens_per_s"],

            "batch8_tokens_per_s":
                b8["tokens_per_s"],
        },
        file,
        indent=2,
    )


print(
    "wrote",
    len(rows),
    "rows to profile.json",
)

print(
    "wrote batch_check.json"
)

wrote 6 rows to profile.json
wrote batch_check.json


In [14]:
!python verify_cell.py

rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 26.4, batch-8 tokens/s: 208.6
GREEN CHECK: PASS
